# Kasus A - Deteksi penipuan pembukaan rekening (BAF Base, month 0-3)

Target: maksimalkan **TPR pada FPR 5%** (penipu tertangkap, nasabah jujur salah tandai <= 5%).

Cara menjalankan: letakkan `Base.csv` (Kaggle BAF, varian Base) di `data/Base.csv`, lalu *Run All*.
Uji cepat tanpa data asli: set env `SMOKE_TEST=1` (data sintetis, **skornya tidak bermakna**).

In [7]:
import os, json
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"
DATA_PATH = os.path.join("data", "Base.csv")
MODEL_DIR = "models"
SEED = 42
TARGET_FPR = 0.05
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Muat data
Hanya baris dengan `month` 0-3 yang dipakai.

In [8]:
def make_synthetic(n=40000, seed=0):
    r = np.random.default_rng(seed)
    df = pd.DataFrame({
        "fraud_bool": (r.random(n) < 0.015).astype(int),
        "income": r.random(n).round(2),
        "name_email_similarity": r.random(n),
        "prev_address_months_count": np.where(r.random(n) < .7, -1, r.integers(0, 200, n)),
        "current_address_months_count": r.integers(-1, 300, n),
        "customer_age": r.choice([10, 20, 30, 40, 50, 60, 70, 80, 90], n),
        "days_since_request": r.random(n) * 3,
        "intended_balcon_amount": np.where(r.random(n) < .75, -1.0, r.random(n) * 50),
        "payment_type": r.choice(list("ABCDE"), n),
        "zip_count_4w": r.integers(1, 6000, n),
        "velocity_6h": r.random(n) * 10000,
        "velocity_24h": r.random(n) * 6000,
        "velocity_4w": r.random(n) * 5000,
        "bank_branch_count_8w": r.integers(0, 2500, n),
        "date_of_birth_distinct_emails_4w": r.integers(0, 40, n),
        "employment_status": r.choice(list("ABCDEFG"), n),
        "credit_risk_score": r.integers(-100, 400, n),
        "email_is_free": r.integers(0, 2, n),
        "housing_status": r.choice(list("ABCDEFG"), n),
        "phone_home_valid": r.integers(0, 2, n),
        "phone_mobile_valid": r.integers(0, 2, n),
        "bank_months_count": np.where(r.random(n) < .25, -1, r.integers(0, 30, n)),
        "has_other_cards": r.integers(0, 2, n),
        "proposed_credit_limit": r.choice([200., 500., 1000., 1500., 2000.], n),
        "foreign_request": r.integers(0, 2, n),
        "source": r.choice(["INTERNET", "TELEAPP"], n),
        "session_length_in_minutes": r.random(n) * 30,
        "device_os": r.choice(["linux", "windows", "macintosh", "x11", "other"], n),
        "keep_alive_session": r.integers(0, 2, n),
        "device_distinct_emails_8w": r.integers(-1, 3, n),
        "device_fraud_count": 0,
        "month": r.integers(0, 8, n),
    })
    f = df.fraud_bool == 1  # sinyal lemah agar pipeline punya sesuatu untuk dipelajari
    df.loc[f, "name_email_similarity"] *= 0.4
    df.loc[f, "credit_risk_score"] += 120
    df.loc[f, "velocity_6h"] += 3000
    return df


if SMOKE_TEST:
    print("SMOKE_TEST: data sintetis, skor tidak bermakna")
    df = make_synthetic()
else:
    df = pd.read_csv(DATA_PATH)

df = df[df["month"] <= 3].reset_index(drop=True)
print(df.shape, "| fraud rate: %.3f%%" % (100 * df.fraud_bool.mean()))
print(df.groupby("month").fraud_bool.agg(["size", "mean"]))

(547975, 32) | fraud rate: 0.965%
         size      mean
month                  
0.0    132440  0.011326
1.0    127620  0.009387
2.0    136979  0.008746
3.0    150936  0.009222


## 2. Pembersihan fitur
Menurut datasheet BAF, **-1 (atau nilai negatif pada `intended_balcon_amount`) berarti data hilang**, bukan angka sungguhan.
Diubah ke `NaN` agar model memperlakukannya sebagai "hilang" (pola hilang sendiri bisa menjadi sinyal).
`device_fraud_count` konstan 0, jadi dibuang.

In [9]:
MISSING_NEG1 = ["prev_address_months_count", "current_address_months_count", "bank_months_count",
                "session_length_in_minutes", "device_distinct_emails_8w"]
CAT_COLS = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
DROP = ["device_fraud_count", "month", "fraud_bool"]


def clean(d):
    d = d.copy()
    for c in MISSING_NEG1:
        d[c] = d[c].where(d[c] != -1, np.nan)
    d["intended_balcon_amount"] = d["intended_balcon_amount"].where(d["intended_balcon_amount"] >= 0, np.nan)
    return d


df = clean(df)
FEATURES = [c for c in df.columns if c not in DROP]
print(len(FEATURES), "fitur")

29 fitur


## 3. Pembagian latih / validasi: berdasarkan waktu
**Latih = month 0-2, validasi = month 3.**

Alasan: bank melatih model dari masa lalu lalu menilai pengajuan yang *akan datang*, dan pola penipuan bergeser antar bulan
(BAF sengaja punya temporal drift). Split acak mencampur bulan yang sama di kedua sisi sehingga skor terlalu optimis;
split temporal memberi estimasi yang lebih jujur untuk kondisi deployment. Satu baris = satu pengajuan, jadi tidak ada
kebocoran baris yang sama antar-split.

In [10]:
tr = df[df["month"] <= 2]
va = df[df["month"] == 3]
X_tr, y_tr = tr[FEATURES], tr["fraud_bool"].values
X_va, y_va = va[FEATURES], va["fraud_bool"].values
print("latih:", X_tr.shape, "fraud %d" % y_tr.sum(), "| validasi:", X_va.shape, "fraud %d" % y_va.sum())

latih: (397039, 29) fraud 3896 | validasi: (150936, 29) fraud 1392


## 4. Model: gradient boosting (HistGradientBoosting)
Data tabular campuran numerik + kategorik dengan banyak nilai hilang dan kelas sangat tidak seimbang (~1% penipu).
Boosting berbasis pohon menangani nilai hilang dan interaksi tanpa scaling, dan hanya butuh scikit-learn.
Karena metriknya berbasis *peringkat* skor (TPR di FPR tetap), tidak dipakai resampling/bobot kelas: keduanya
mengubah kalibrasi tetapi jarang membantu peringkat, dan threshold ditentukan langsung dari FPR 5%.

In [11]:
pre = ColumnTransformer(
    [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                            encoded_missing_value=np.nan), CAT_COLS)],
    remainder="passthrough",
)
clf = HistGradientBoostingClassifier(
    learning_rate=0.05, max_iter=500, max_leaf_nodes=31, min_samples_leaf=100,
    l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=30, categorical_features=list(range(len(CAT_COLS))), random_state=SEED,
)
model = Pipeline([("pre", pre), ("clf", clf)])
model.fit(X_tr, y_tr)
print("iterasi terpakai:", model.named_steps["clf"].n_iter_)

iterasi terpakai: 219


## 5. Evaluasi: TPR pada FPR 5% (data validasi)

In [12]:
def tpr_at_fpr(y_true, score, max_fpr=0.05):
    fpr, tpr, thr = roc_curve(y_true, score)
    ok = fpr <= max_fpr
    i = np.where(ok)[0][-1]
    return tpr[i], fpr[i], thr[i]


p_va = model.predict_proba(X_va)[:, 1]
tpr, fpr, thr = tpr_at_fpr(y_va, p_va, TARGET_FPR)
auc = roc_auc_score(y_va, p_va)
print("TPR @ FPR<=5%%: %.4f (FPR sebenarnya %.4f, threshold %.5f)" % (tpr, fpr, thr))
print("ROC-AUC validasi: %.4f" % auc)

TPR @ FPR<=5%: 0.5560 (FPR sebenarnya 0.0500, threshold 0.05355)
ROC-AUC validasi: 0.9123


## 6. Simpan model dan muat ulang
Threshold (skor yang memberi FPR <= 5% di validasi) disimpan bersama model.

In [13]:
MODEL_PATH = os.path.join(MODEL_DIR, "fraud_model.joblib")
META_PATH = os.path.join(MODEL_DIR, "fraud_model_meta.json")
joblib.dump(model, MODEL_PATH)
meta = {"features": FEATURES, "threshold": float(thr), "target_fpr": TARGET_FPR,
        "val_tpr_at_fpr5": float(tpr), "val_auc": float(auc), "smoke_test": SMOKE_TEST}
with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

loaded = joblib.load(MODEL_PATH)
assert np.allclose(loaded.predict_proba(X_va)[:, 1], p_va), "model muat-ulang beda hasil"
print("model tersimpan & berhasil dimuat ulang:", MODEL_PATH)


def score_applications(raw_df):
    # raw_df: pengajuan baru dengan kolom seperti Base.csv; kembalikan skor + flag
    m = joblib.load(MODEL_PATH)
    meta_ = json.load(open(META_PATH))
    x = clean(raw_df)[meta_["features"]]
    s = m.predict_proba(x)[:, 1]
    return pd.DataFrame({"fraud_score": s, "flag": s >= meta_["threshold"]}, index=raw_df.index)


print(score_applications(va.head(5)))

model tersimpan & berhasil dimuat ulang: models/fraud_model.joblib
        fraud_score   flag
397039     0.479344   True
397040     0.000638  False
397041     0.000593  False
397042     0.001317  False
397043     0.002486  False
